In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from matplotlib.animation import FuncAnimation
from IPython.display import HTML
rcParams.update({'figure.autolayout': True})
import os
import IPython

In [ ]:
GRAV = 4 * np.pi**2 #AU^3/M_sun/yr^2

#acceleration due to gravity
def g_accel(m,x,y):
    r =  np.sqrt(x**2 + y**2)
    GM = GRAV * m
    ax = -GM * x / r**3
    ay = -GM * y / r**3
    return ax, ay


## Earth!

In [64]:
#parameters to change
theta_e = np.pi *0   #earth position in radian
dt = 0.01 #year
total_t = 3 #year
v_factor = 0.8


#initial conditions
M_sun  = 1 #M_sun
steps = int(total_t / dt)
t = np.arange(steps + 1) * dt


#parameter for Eulers Method
x_e = np.zeros(steps + 1)
y_e = np.zeros(steps + 1)
vx_e = np.zeros(steps + 1)
vy_e = np.zeros(steps + 1)

#initial value
x_e[0] = 1* np.cos(theta_e)#AU
y_e[0] = 1* np.sin(theta_e)
vx_e[0] = -2 * np.pi * np.sin(theta_e) * v_factor
vy_e[0] = 2 * np.pi * np.cos(theta_e) * v_factor #AU/year

#Eulers Method
for n in range(steps):
    ax,ay = g_accel(M_sun,x_e[n],y_e[n])

    vx_e[n+1] = vx_e[n] + dt * ax
    vy_e[n+1] = vy_e[n] + dt * ay

    x_e[n+1] = x_e[n] + dt * vx_e[n] #using old value of v per Euler
    y_e[n+1] = y_e[n] + dt * vy_e[n]
      

#some other parameters of interest
speed_e = np.sqrt(vx_e**2 + vy_e**2)
r_e = np.sqrt(x_e**2 + y_e**2)
energy_e = 0.5 * speed_e**2 - GRAV*M_sun / r_e



In [65]:
#parameter for Leapfrog Method
x_l = np.zeros(steps + 1)
y_l = np.zeros(steps + 1)
vx_l = np.zeros(steps + 1)
vy_l = np.zeros(steps + 1)

#initial value
x_l[0] = 1* np.cos(theta_e)#AU
y_l[0] = 1* np.sin(theta_e)
vx_l[0] =  -2 * np.pi * np.sin(theta_e) * v_factor
vy_l[0] =2 * np.pi * np.cos(theta_e) * v_factor #AU/year
#Leapfrog method
for n in range(steps):
    ax,ay = g_accel(M_sun,x_l[n],y_l[n])

    # kick: half-step velocity
    vx_half = vx_l[n] + 0.5 * dt * ax
    vy_half = vy_l[n] + 0.5 * dt * ay

    # drift: full-step position
    x_l[n+1] = x_l[n] + dt * vx_half
    y_l[n+1] = y_l[n] + dt * vy_half

    # acceleration at new position
    ax_new, ay_new = g_accel(M_sun,x_l[n+1], y_l[n+1])

    # kick: finish velocity step
    vx_l[n+1] = vx_half + 0.5 * dt * ax_new
    vy_l[n+1] = vy_half + 0.5 * dt * ay_new

speed_l = np.sqrt(vx_l**2 + vy_l**2)
r_l = np.sqrt(x_l**2 + y_l**2)
energy_l = 0.5 * speed_l**2 - GRAV*M_sun / r_l

In [ ]:
#learning from Matplotlib's animation page
frame_step = 5          # show every 5th point

fig = plt.figure(figsize=(10, 6))
gs = fig.add_gridspec(2, 2, width_ratios=[2, 1], height_ratios=[1, 1])

ax_orbit = fig.add_subplot(gs[:, 0])   
ax_energye = fig.add_subplot(gs[0, 1]) 
ax_energyl= fig.add_subplot(gs[1, 1])   

ax_orbit.plot(0, 0, 'o', c= "red" ,markersize=12, label='Sun')
circ = patches.Circle((0, 0), 1, fill = 0, ec = "black", ls = '--')
ax_orbit.add_patch(circ)
ax_orbit.set_aspect('equal')
ax_orbit.set_xlim(-5,5)
ax_orbit.set_ylim(-5,5)
ax_orbit.set_xlabel("x [AU]",fontsize = 20)
ax_orbit.set_ylabel("y [AU]",fontsize = 20)
ax_orbit.set_title("Orbit with 0.8 VE",fontsize = 25)

#objects
time = ax_orbit.text(0.02, 0.95, '', transform=ax_orbit.transAxes,fontsize = 15)

trail_e, = ax_orbit.plot(x_e[0], y_e[0], '-', c = "green", lw=1, label = 'Euler')
earth_e, = ax_orbit.plot(x_e[0], y_e[0], 'o', c = "blue", markersize=6)

trail_l, = ax_orbit.plot(x_l[0], y_l[0], '-', c= "orange", lw=1, label = 'Leapfrog')
earth_l, = ax_orbit.plot(x_l[0], y_l[0], 'o', c = "blue", markersize=6)
ax_orbit.legend()

#energy space for euler
et_e, = ax_energye.plot(t[0],energy_e[0], '-', c = "green", lw=1)
ax_energye.set_xlim(0,max(t))
ax_energye.set_ylim(min(energy_e),max(energy_e))
ax_energye.set_ylabel(r"$\epsilon$",fontsize = 20)
ax_energye.set_title("Energy Varitaion",fontsize = 25)

#energy space for leapfrog
et_l, = ax_energyl.plot(t[0],energy_e[0], '-', c = "orange", lw=1)
ax_energyl.set_xlim(0,max(t))
ax_energyl.set_ylim(min(energy_l),max(energy_l))
ax_energyl.set_ylabel(r"$\epsilon$",fontsize = 20)
ax_energyl.set_xlabel("t (yr)",fontsize = 20)




frames = range(0, len(t), frame_step)

def update(frame):
    earth_e.set_data([x_e[frame]], [y_e[frame]])
    trail_e.set_data(x_e[:frame+1], y_e[:frame+1])

    earth_l.set_data([x_l[frame]], [y_l[frame]])
    trail_l.set_data(x_l[:frame+1], y_l[:frame+1])

    et_e.set_data(t[:frame+1],energy_e[:frame+1])
    et_l.set_data(t[:frame+1],energy_l[:frame+1])


    time.set_text(f"t = {t[frame]:.2f} yr")
    return earth_e, trail_e, time, earth_l, trail_l, et_e, et_l

ani = FuncAnimation(fig,update,frames=frames,blit=True,interval=30)

plt.close(fig) 
#ani.save("orbit_0.8_energy.gif", writer="pillow", fps=20)
HTML(ani.to_jshtml())

In [ ]:
#learning from Matplotlib's animation page
frame_step = 5          # show every 5th point

fig = plt.figure(figsize=(10, 6))
gs = fig.add_gridspec(2, 2, width_ratios=[2, 1], height_ratios=[1, 1])

ax_orbit = fig.add_subplot(gs[:, 0])   
ax_speede = fig.add_subplot(gs[0, 1]) 
ax_speedl= fig.add_subplot(gs[1, 1])   

ax_orbit.plot(0, 0, 'o', c= "red" ,markersize=12, label='Sun')
circ = patches.Circle((0, 0), 1, fill = 0, ec = "black", ls = '--')
ax_orbit.add_patch(circ)
ax_orbit.set_aspect('equal')
ax_orbit.set_xlim(-5,5)
ax_orbit.set_ylim(-5,5)
ax_orbit.set_xlabel("x [AU]",fontsize = 20)
ax_orbit.set_ylabel("y [AU]",fontsize = 20)
ax_orbit.set_title("Orbit with 0.8 VE",fontsize = 25)

#objects
time = ax_orbit.text(0.02, 0.95, '', transform=ax_orbit.transAxes,fontsize = 15)

trail_e, = ax_orbit.plot(x_e[0], y_e[0], '-', c = "green", lw=1, label = 'Euler')
earth_e, = ax_orbit.plot(x_e[0], y_e[0], 'o', c = "blue", markersize=6)

trail_l, = ax_orbit.plot(x_l[0], y_l[0], '-', c= "orange", lw=1, label = 'Leapfrog')
earth_l, = ax_orbit.plot(x_l[0], y_l[0], 'o', c = "blue", markersize=6)
ax_orbit.legend()

#speed space for euler
et_e, = ax_speede.plot(t[0],speed_e[0], '-', c = "green", lw=1)
ax_speede.set_xlim(0,max(t))
ax_speede.set_ylim(min(speed_e),max(speed_e))
ax_speede.set_ylabel("speed (AU/yr)",fontsize = 20)
ax_speede.set_title("Speed Varitaion",fontsize = 25)

#speed space for leapfrog
et_l, = ax_speedl.plot(t[0],speed_e[0], '-', c = "orange", lw=1)
ax_speedl.set_xlim(0,max(t))
ax_speedl.set_ylim(min(speed_l),max(speed_l))
ax_speedl.set_ylabel("speed (AU/yr)",fontsize = 20)
ax_speedl.set_xlabel("t (yr)",fontsize = 20)

frames = range(0, len(t), frame_step)

def update(frame):
    earth_e.set_data([x_e[frame]], [y_e[frame]])
    trail_e.set_data(x_e[:frame+1], y_e[:frame+1])

    earth_l.set_data([x_l[frame]], [y_l[frame]])
    trail_l.set_data(x_l[:frame+1], y_l[:frame+1])

    et_e.set_data(t[:frame+1],speed_e[:frame+1])
    et_l.set_data(t[:frame+1],speed_l[:frame+1])


    time.set_text(f"t = {t[frame]:.2f} yr")
    return earth_e, trail_e, time, earth_l, trail_l, et_e, et_l

ani = FuncAnimation(fig,update,frames=frames,blit=True,interval=30)

plt.close(fig) 
#ani.save("orbit_0.8_speed.gif", writer="pillow", fps=20)
HTML(ani.to_jshtml())

## Voyager!

In [87]:
GRAV = 4 * np.pi**2 #AU^3/M_sun/yr^2

#acceleration due to gravity for multiple objects
def g_accel_multi(m_array, x_array, y_array, position):
    x, y = position
    ax = 0.0
    ay = 0.0

    for i, m_val in enumerate(m_array):
        dx = x - x_array[i]
        dy = y - y_array[i]
        r = np.sqrt(dx**2 + dy**2)

        GM = GRAV * m_val
        ax -= GM * dx / r**3
        ay -= GM * dy / r**3
    return ax, ay

In [108]:
#parameters to change
theta_J = 94.5 * np.pi/180  #Jupiter's position
theta_V = 0 * np.pi/180  #Voyager's position
dt = 0.001 #year
total_t = 3 #year


#initial conditions
M_sun  = 1 #Mass of sun
M_jup  = 0.00095 #Mass of jupiter
steps = int(total_t / dt)
t = np.arange(steps + 1) * dt


#parameter for Jupiter
x_J = np.zeros(steps + 1)
y_J = np.zeros(steps + 1)
vx_J = np.zeros(steps + 1)
vy_J = np.zeros(steps + 1)
x_J[0] = 5.2* np.cos(theta_J)#AU
y_J[0] = 5.2* np.sin(theta_J)
vx_J[0] =  -2 /np.sqrt(5.2)* np.pi * np.sin(theta_J)
vy_J[0] =2 * np.pi /np.sqrt(5.2)* np.cos(theta_J)#AU/year

#parameter for Voyager
x_V = np.zeros(steps + 1)
y_V = np.zeros(steps + 1)
vx_V = np.zeros(steps + 1)
vy_V = np.zeros(steps + 1)
x_V[0] = 1* np.cos(theta_V)#AU
y_V[0] = 1* np.sin(theta_V)
vx_V[0] =  -8.88* np.sin(theta_V) # 42.1 km/s or 8.88 AU/year
vy_V[0] =   8.88* np.cos(theta_V)


#Leapfrog method
for n in range(steps):
    ax,ay = g_accel(M_sun,x_J[n],y_J[n])
    ax_V,ay_V = g_accel_multi([M_sun,M_jup], [0,x_J[n]],[0,y_J[n]],[x_V[n],y_V[n]])

    # kick: half-step velocity
    vx_half = vx_J[n] + 0.5 * dt * ax
    vy_half = vy_J[n] + 0.5 * dt * ay
    vx_half_V = vx_V[n] + 0.5 * dt * ax_V
    vy_half_V = vy_V[n] + 0.5 * dt * ay_V

    # drift: full-step position
    x_J[n+1] = x_J[n] + dt * vx_half
    y_J[n+1] = y_J[n] + dt * vy_half
    x_V[n+1] = x_V[n] + dt * vx_half_V
    y_V[n+1] = y_V[n] + dt * vy_half_V

    # acceleration at new position
    ax_new, ay_new = g_accel(M_sun,x_J[n+1], y_J[n+1])
    ax_new_V,ay_new_V = g_accel_multi([M_sun,M_jup], [0,x_J[n+1]],[0,y_J[n+1]],[x_V[n+1],y_V[n+1]])

    # kick: finish velocity step
    vx_J[n+1] = vx_half + 0.5 * dt * ax_new
    vy_J[n+1] = vy_half + 0.5 * dt * ay_new
    vx_V[n+1] = vx_half_V + 0.5 * dt * ax_new_V
    vy_V[n+1] = vy_half_V + 0.5 * dt * ay_new_V

#side charts
dist_JV = np.sqrt((x_V-x_J)**2 + (y_V-y_J)**2)
speed_V = np.sqrt(vx_V**2 + vy_V**2)

i_min = np.argmin(dist_JV)
print(f"Closest distance = {dist_JV[i_min]:.2f} AU")
print(f"Time of closest approach = {t[i_min]:.2f} yr")

Closest distance = 0.00 AU
Time of closest approach = 1.10 yr


In [ ]:
frame_step = 20          # skipping frames for faster animation renders

fig = plt.figure(figsize=(10, 6))
gs = fig.add_gridspec(2, 2, width_ratios=[2, 1], height_ratios=[1, 1])

ax_orbit = fig.add_subplot(gs[:, 0])   
ax_dist = fig.add_subplot(gs[0, 1]) 
ax_vel= fig.add_subplot(gs[1, 1])   

ax_orbit.plot(0, 0, 'o', c= "red" ,markersize=12, label='Sun')
circ_J = patches.Circle((0, 0), 5.2, fill = 0, ec = "black", ls = '--')
ax_orbit.add_patch(circ_J)
circ_E = patches.Circle((0, 0), 1, fill = 0, ec = "black", ls = '--')
ax_orbit.add_patch(circ_E)

ax_orbit.set_aspect('equal')
ax_orbit.set_xlim(-10,10)
ax_orbit.set_ylim(-10,10)
ax_orbit.set_xlabel("x [AU]",fontsize = 20)
ax_orbit.set_ylabel("y [AU]",fontsize = 20)
ax_orbit.set_title("Voyager's Journey",fontsize = 25)

#objects
time = ax_orbit.text(0.02, 0.95, '', transform=ax_orbit.transAxes,fontsize = 15)
jupiter_J, = ax_orbit.plot(x_J[0], y_J[0], 'o', c = "Orange", markersize=9, label = 'Jupiter')
trail_V, = ax_orbit.plot(x_V[0], y_V[0], '-', c = "green", lw=1, label = 'Voyager')
Voyager, = ax_orbit.plot(x_V[0], y_V[0], 'o', c = "blue", markersize=3)

ax_orbit.legend()

#distance between Voyager and Jupiter
dista_e, = ax_dist.plot(t[0],dist_JV[0], '-', c = "orange", lw=1)
ax_dist.set_xlim(0,max(t))
ax_dist.set_ylim(0,max(dist_JV))
ax_dist.set_ylabel("distance(AU)",fontsize = 20)
ax_dist.set_title("Distance to Jupiter",fontsize = 25)

#velocity for Voyager

velo_V, = ax_vel.plot(t[0],speed_V[0], '-', c = "green", lw=1)
ax_vel.set_xlim(0,max(t))
ax_vel.set_ylim(0,max(speed_V))
ax_vel.set_ylabel("Speed (AU/yr)",fontsize = 20)
ax_vel.set_xlabel("t (yr)",fontsize = 20)
ax_vel.set_title("Velocity of Voyager",fontsize = 20)



frames = range(0, len(t), frame_step)

def update(frame):

    jupiter_J.set_data([x_J[frame]], [y_J[frame]])
    Voyager.set_data([x_V[frame]], [y_V[frame]])
    trail_V.set_data(x_V[:frame+1], y_V[:frame+1])

    dista_e.set_data(t[:frame+1],dist_JV[:frame+1])
    velo_V.set_data(t[:frame+1],speed_V[:frame+1])


    years = int(t[frame])
    months = (t[frame] - years) * 12
    time.set_text(f"t = {years} yr {months:.1f} months")
    
    return time, jupiter_J, Voyager, trail_V, dista_e,velo_V

ani = FuncAnimation(fig,update,frames=frames,blit=True,interval=30)

plt.close(fig) 
#ani.save("Voyager_w_Jupiter_bounce_back.gif", writer="pillow", fps=20)
HTML(ani.to_jshtml())